In [18]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import folium

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries" / "processed"
JOBS_DIR = DATA_DIR / "jobs" / "lodes_processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

In [19]:
TOC_GPKG = BOUNDARIES_DIR / "toc_clean.gpkg"
VILLAGE_GPKG = BOUNDARIES_DIR / "village_clean.gpkg"
TOC_STATS_CSV = JOBS_DIR / "toc_jobs_2022.csv"
VILLAGE_STATS_CSV = JOBS_DIR / "village_jobs_2022.csv"

In [24]:
villages = gpd.read_file(VILLAGE_GPKG)

village_stats = pd.read_csv(VILLAGE_STATS_CSV)

village_merge = villages.merge(
    village_stats,
    on="NAME",
    how="left"
)

In [25]:
toc = gpd.read_file(TOC_GPKG)

toc_stats = pd.read_csv(TOC_STATS_CSV)

toc_merge = toc.merge(
    toc_stats,
    on="OBJECTID",
    how="left"
)

In [27]:
village_merge.head()

,OBJECTID_x,NAME,geometry_x,Village_ID,OBJECTID_y,geometry_y,jobs_total,acres_contributing,village_area_sqft,village_acres,jobs_per_acre,jobs_per_acre_label
0,16,Ahwatukee Foothills,"POLYGON ((683945.25 859952.062, 683930.25 8598...",0,16,"POLYGON ((683945.25 859952.0620078743, 683930....",31854.214182,22832.880975,9.946003e+08,22832.880975,1.395103,1.40
1,17,Laveen,"POLYGON ((638589.25 843376, 629247.625 838551....",1,17,"POLYGON ((638589.25 843376, 629247.625 838551....",38138.709412,19579.686598,8.528911e+08,19579.686598,1.947871,1.95
2,18,South Mountain,"POLYGON ((683945.25 859952.062, 683530.875 860...",2,18,"POLYGON ((683945.25 859952.0620078743, 683530....",60064.332489,25481.856132,1.109990e+09,25481.856132,2.357141,2.36
3,19,Estrella,"POLYGON ((607040.694 890232.667, 607046.417 89...",3,19,"POLYGON ((607040.6938976385 890232.6669947505,...",20731.840319,26503.624912,1.154498e+09,26503.624912,0.782227,0.78
4,20,Central City,"POLYGON ((669683.439 897001.625, 672160.625 89...",4,20,"POLYGON ((669683.4389763772 897001.625, 672160...",33900.231265,13600.592409,5.924418e+08,13600.592409,2.492555,2.49


In [28]:
toc_merge.head()

,OBJECTID,APPLICABIL_x,geometry_x,TOC_ID,APPLICABIL_y,geometry_y,jobs_total,acres_contributing,toc_area_sqft,toc_acres,jobs_per_acre,jobs_per_acre_label
0,1,TOD District - Gateway,"POLYGON ((663431.678 894322.427, 663434.469 89...",0,TOD District - Gateway,"POLYGON ((663431.6781360433 894322.4273211062,...",4984.867123,2418.944362,1.053692e+08,2418.944362,2.060761,2.06
1,2,TOD District - Eastlake Garfield,"POLYGON ((654751.939 894390.665, 654764.736 89...",1,TOD District - Eastlake Garfield,"POLYGON ((654751.9393232353 894390.6651817411,...",6847.046904,1220.462979,5.316337e+07,1220.462979,5.610205,5.61
2,3,TOD District - Midtown,"POLYGON ((654767.886 907514.625, 654767.631 90...",2,TOD District - Midtown,"POLYGON ((654767.8861735687 907514.6247785166,...",7099.922548,1283.000217,5.588749e+07,1283.000217,5.533844,5.53
3,4,TOD District - Uptown,"POLYGON ((649448.684 915530.44, 654791.762 915...",3,TOD District - Uptown,"POLYGON ((649448.6837655641 915530.440365456, ...",441.878230,1332.064285,5.802472e+07,1332.064285,0.331724,0.33
4,5,TOD District - Solano,"POLYGON ((646830.036 919545.608, 646829.436 91...",4,TOD District - Solano,"POLYGON ((646830.0364656076 919545.6076733321,...",2006.609086,1100.265433,4.792756e+07,1100.265433,1.823750,1.82


In [32]:
# Somehow these are x and y geometry, so time to fix that -

# Fix village geometry
if "geometry_x" in village_merge.columns:
    village_merge = village_merge.set_geometry("geometry_x")
    village_merge = village_merge.rename_geometry("geometry")
    if "geometry_y" in village_merge.columns:
        village_merge = village_merge.drop(columns=["geometry_y"])

# Fix TOC geometry
if "geometry_x" in toc_merge.columns:
    toc_merge = toc_merge.set_geometry("geometry_x")
    toc_merge = toc_merge.rename_geometry("geometry")
    if "geometry_y" in toc_merge.columns:
        toc_merge = toc_merge.drop(columns=["geometry_y"])

# fixing APPLICABIL since that's also x and y:
if "APPLICABIL_x" in toc_merge.columns:
    toc_merge["APPLICABIL"] = toc_merge["APPLICABIL_x"]
    toc_merge = toc_merge.drop(columns=["APPLICABIL_x"])
    if "APPLICABIL_y" in toc_merge.columns:
        toc_merge = toc_merge.drop(columns=["APPLICABIL_y"])

# Reprojecting CRS just in case
if village_merge.crs is None:
    village_merge = village_merge.set_crs(2868)
if toc_merge.crs is None:
    toc_merge = toc_merge.set_crs(2868)

In [33]:
toc_merge.head()

,OBJECTID,geometry,TOC_ID,jobs_total,acres_contributing,toc_area_sqft,toc_acres,jobs_per_acre,jobs_per_acre_label,APPLICABIL
0,1,"POLYGON ((663431.678 894322.427, 663434.469 89...",0,4984.867123,2418.944362,1.053692e+08,2418.944362,2.060761,2.06,TOD District - Gateway
1,2,"POLYGON ((654751.939 894390.665, 654764.736 89...",1,6847.046904,1220.462979,5.316337e+07,1220.462979,5.610205,5.61,TOD District - Eastlake Garfield
2,3,"POLYGON ((654767.886 907514.625, 654767.631 90...",2,7099.922548,1283.000217,5.588749e+07,1283.000217,5.533844,5.53,TOD District - Midtown
3,4,"POLYGON ((649448.684 915530.44, 654791.762 915...",3,441.878230,1332.064285,5.802472e+07,1332.064285,0.331724,0.33,TOD District - Uptown
4,5,"POLYGON ((646830.036 919545.608, 646829.436 91...",4,2006.609086,1100.265433,4.792756e+07,1100.265433,1.823750,1.82,TOD District - Solano


In [ ]:
# Formatting labels for maps:

village_merge["jobs_per_acre_label"] = village_merge["jobs_per_acre"].map(
    lambda v: f"{v:,.2f}" if pd.notnull(v) else "N/A"
)
toc_merge["jobs_per_acre_label"] = toc_merge["jobs_per_acre"].map(
    lambda v: f"{v:,.2f}" if pd.notnull(v) else "N/A"
)

# --- Reproject for web mapping ---
vill_web = village_merge.to_crs(4326).copy()
toc_web  = toc_merge.to_crs(4326).copy()

# ----------------------------------------------------
# Base map: villages by jobs_per_acre
# ----------------------------------------------------
m = vill_web.explore(
    column="jobs_per_acre",
    cmap="Blues",
    legend=True,
    legend_kwds={"caption": "Village Jobs per Acre (Gross)"},
    style_kwds={
        "fillOpacity": 0.35,
        "weight": 0.5,
        "color": "black",
    },
    tooltip=[
        "NAME",
        "jobs_total",
        "village_acres",
        "jobs_per_acre_label",
    ],
    name="Villages (jobs per acre)",
)

# ----------------------------------------------------
# Overlay: TOCs by jobs_per_acre
# ----------------------------------------------------
toc_web.explore(
    m=m,
    column="jobs_per_acre",
    cmap="viridis",
    legend=True,
    style_kwds={
        "fillOpacity": 0.85,
        "weight": 0.5,
        "color": "black",
    },
    tooltip=[
        "APPLICABIL",
        "jobs_total",
        "toc_acres",
        "jobs_per_acre_label",
    ],
    name="TOCs (jobs per acre)",
)

# Optional layer toggle
folium.LayerControl(collapsed=False).add_to(m)

# Save to HTML
out_path = OUTPUTS_DIR / "tocs_vs_villages_jobs_per_acre.html"
m.save(out_path)

print("Saved:", out_path)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\tocs_vs_villages_jobs_per_acre.html
